# trajectory-kit: Tutorial

This notebook demonstrates the core concepts of `trajectory-kit` using a 20-atom synthetic system so it runs on any machine without real simulation data.

---
## System overview

| Segment | Residues | Atoms | Description |
|---------|----------|-------|-------------|
| `PROT`  | ALA (1), GLY (2), PRO (3) | 14 | Protein backbone + ALA sidechain |
| `SOLV`  | TIP (4), TIP (5)           | 6  | Two TIP3 water molecules |

**Bond graph** — 17 bonds total, degree distribution: 10 terminal (degree-1), 6 bridging (degree-2), 4 branching (degree-3).

---
## Concepts

Every simulation is represented by up to three files loaded into a `sim` object:

| Domain | Role | Supported formats |
|--------|------|-------------------|
| **Typing** | Atom names, coordinates, occupancy | `.pdb`, `.xyz`, `.mae` |
| **Topology** | Bond graph, charges, masses | `.psf`, `.mae` |
| **Trajectory** | Positions over time | `.dcd` |

Queries always follow the same **include / exclude** schema and always return `list[int]` global atom indices, which feed directly into trajectory reads.


## 0. Setup — write synthetic files to disk

In [1]:
import struct
import tempfile
from pathlib import Path
import numpy as np

tmp = Path(tempfile.mkdtemp())

# ── 20-atom synthetic system ──────────────────────────────────────────────
# Protein (PROT): ALA(6) + GLY(5) + PRO(3) = 14 atoms, 3 residues
# Solvent (SOLV): 2 x TIP3 water molecules = 6 atoms, 2 residues
#
# Bond graph (17 bonds):
#   Backbone: N-CA-C chain with peptide bonds
#   ALA sidechain: CA-CB
#   Carbonyl: C=O on each residue
#   PRO N closes the chain
#   Waters: each OH2 bonded to H1 and H2
#
# Degree distribution: 10 x degree-1, 6 x degree-2, 4 x degree-3
# Atom types used: NH1, H, CT1, CT3, C, O, CT2, N, CP1, OT, HT

pdb_text = """\
ATOM      1  N   ALA A   1       1.000   5.000   5.000  1.00  0.00      PROT
ATOM      2  HN  ALA A   1       0.100   5.800   5.300  1.00  0.00      PROT
ATOM      3  CA  ALA A   1       2.400   5.000   5.000  1.00  0.00      PROT
ATOM      4  CB  ALA A   1       3.000   6.300   5.200  1.00  0.00      PROT
ATOM      5  C   ALA A   1       3.200   3.900   4.600  1.00  0.00      PROT
ATOM      6  O   ALA A   1       2.800   2.800   4.400  1.00  0.00      PROT
ATOM      7  N   GLY A   2       4.600   4.100   4.500  1.00  0.00      PROT
ATOM      8  HN  GLY A   2       5.000   5.000   4.700  1.00  0.00      PROT
ATOM      9  CA  GLY A   2       5.800   3.200   4.100  1.00  0.00      PROT
ATOM     10  C   GLY A   2       7.200   3.500   4.000  1.00  0.00      PROT
ATOM     11  O   GLY A   2       7.700   4.600   4.200  1.00  0.00      PROT
ATOM     12  N   PRO A   3       8.100   2.500   3.700  1.00  0.00      PROT
ATOM     13  CA  PRO A   3       9.500   2.800   3.600  1.00  0.00      PROT
ATOM     14  C   PRO A   3      10.300   1.900   3.200  1.00  0.00      PROT
ATOM     15  OH2 TIP B   4      12.000   5.000   5.000  1.00  0.00      SOLV
ATOM     16  H1  TIP B   4      12.900   5.500   5.100  1.00  0.00      SOLV
ATOM     17  H2  TIP B   4      11.300   5.700   4.800  1.00  0.00      SOLV
ATOM     18  OH2 TIP B   5      14.000   3.000   5.500  1.00  0.00      SOLV
ATOM     19  H1  TIP B   5      14.800   3.600   5.400  1.00  0.00      SOLV
ATOM     20  H2  TIP B   5      13.400   3.700   5.200  1.00  0.00      SOLV
END
"""

psf_text = """\
PSF

       1 !NTITLE
 REMARKS synthetic 20-atom ALA-GLY-PRO + 2xTIP3

      20 !NATOM
       1 PROT     1 ALA  N    NH1     -0.47000     14.007           0
       2 PROT     1 ALA  HN   H        0.31000      1.008           0
       3 PROT     1 ALA  CA   CT1     -0.02000     12.011           0
       4 PROT     1 ALA  CB   CT3     -0.27000     12.011           0
       5 PROT     1 ALA  C    C        0.51000     12.011           0
       6 PROT     1 ALA  O    O       -0.51000     15.999           0
       7 PROT     2 GLY  N    NH1     -0.47000     14.007           0
       8 PROT     2 GLY  HN   H        0.31000      1.008           0
       9 PROT     2 GLY  CA   CT2     -0.18000     12.011           0
      10 PROT     2 GLY  C    C        0.51000     12.011           0
      11 PROT     2 GLY  O    O       -0.51000     15.999           0
      12 PROT     3 PRO  N    N       -0.29000     14.007           0
      13 PROT     3 PRO  CA   CP1     -0.02000     12.011           0
      14 PROT     3 PRO  C    C        0.51000     12.011           0
      15 SOLV     4 TIP  OH2  OT      -0.83400     15.999           0
      16 SOLV     4 TIP  H1   HT       0.41700      1.008           0
      17 SOLV     4 TIP  H2   HT       0.41700      1.008           0
      18 SOLV     5 TIP  OH2  OT      -0.83400     15.999           0
      19 SOLV     5 TIP  H1   HT       0.41700      1.008           0
      20 SOLV     5 TIP  H2   HT       0.41700      1.008           0

      17 !NBOND: bonds
       1       2       1       3       3       4       3       5
       5       6       5       7       7       8       7       9
       9      10      10      11      10      12      12      13
      13      14      15      16      15      17      18      19
      18      20

       0 !NTHETA: angles

       0 !NPHI: dihedrals

       0 !NIMPHI: impropers

"""

def write_dcd(path, n_atoms=20, n_frames=5):
    """5-frame DCD. Each frame shifts all atoms by +1 Å along x."""
    def record(payload):
        n = struct.pack('<i', len(payload))
        return n + payload + n
    icntrl = [0] * 20
    icntrl[0] = n_frames
    hdr = b'CORD' + struct.pack('<20i', *icntrl)
    hdr += b'\x00' * (84 - len(hdr))
    title = struct.pack('<i', 1) + b'synthetic 20-atom trajectory    '
    natom = struct.pack('<i', n_atoms)
    # Base coords from PDB (x values)
    x0 = np.array([ 1.0, 0.1, 2.4, 3.0, 3.2, 2.8,
                    4.6, 5.0, 5.8, 7.2, 7.7, 8.1,
                    9.5,10.3,12.0,12.9,11.3,14.0,14.8,13.4], dtype=np.float32)
    y0 = np.array([ 5.0, 5.8, 5.0, 6.3, 3.9, 2.8,
                    4.1, 5.0, 3.2, 3.5, 4.6, 2.5,
                    2.8, 1.9, 5.0, 5.5, 5.7, 3.0, 3.6, 3.7], dtype=np.float32)
    z0 = np.array([ 5.0, 5.3, 5.0, 5.2, 4.6, 4.4,
                    4.5, 4.7, 4.1, 4.0, 4.2, 3.7,
                    3.6, 3.2, 5.0, 5.1, 4.8, 5.5, 5.4, 5.2], dtype=np.float32)
    with open(path, 'wb') as f:
        f.write(record(hdr))
        f.write(record(title))
        f.write(record(natom))
        for fi in range(n_frames):
            xs = x0 + float(fi)
            f.write(record(xs.tobytes()))
            f.write(record(y0.tobytes()))
            f.write(record(z0.tobytes()))

pdb_path = tmp / 'synthetic.pdb'
psf_path = tmp / 'synthetic.psf'
dcd_path = tmp / 'synthetic.dcd'

pdb_path.write_text(pdb_text, encoding='ascii')
psf_path.write_text(psf_text, encoding='ascii')
write_dcd(dcd_path)

print('Synthetic files written to:', tmp)
print('  PDB: 20 atoms (ALA-GLY-PRO backbone + 2x TIP3 water)')
print('  PSF: 20 atoms, 17 bonds, 5 residues, 2 segments (PROT/SOLV)')
print('  DCD: 5 frames — x coordinates shift +1 Å per frame')


Synthetic files written to: C:\Users\zhikh\AppData\Local\Temp\tmprnn4rk6y
  PDB: 20 atoms (ALA-GLY-PRO backbone + 2x TIP3 water)
  PSF: 20 atoms, 17 bonds, 5 residues, 2 segments (PROT/SOLV)
  DCD: 5 frames — x coordinates shift +1 Å per frame


## 1. Loading a sim

Create a `sim` object by pointing it at your files. Any combination is valid — you do not need all three.
Call `print_info()` to see loaded files, system properties, and every keyword/request available for each domain.


In [2]:
from trajectory_kit import sim as Sim

s = Sim(
    typing=pdb_path,
    topology=psf_path,
    trajectory=dcd_path,
)

s.print_info()



=== SIMULATION INFO ===

  files
    typing     synthetic.pdb  (.pdb)
    topology   synthetic.psf  (.psf)
    trajectory synthetic.dcd  (.dcd)

  system properties
    num_atoms        20
    num_residues     5
    num_frames       5
    start_box_size   (0.1, 14.8, 1.9, 6.3, 3.2, 5.5)

  available keywords and requests
  ──────────  ───────────────────────────────  ──────────────────────────  ────────────────
              typing                           topology                    trajectory      
  ──────────  ───────────────────────────────  ──────────────────────────  ────────────────
  keywords    atom_name                        atom_name                   frame_interval  
              global_ids                       atom_type                   global_ids      
              local_ids                        bonded_with                                 
              occupancy                        bonded_with_mode                            
              residue_ids       

## 2. The Query Schema

Every field in a query follows the same include/exclude pattern.

**Categorical fields** (atom name, residue name, segment name, atom type):
```python
"atom_name": include_set                     # bare set — include only
"atom_name": (include_set, exclude_set)      # explicit two-sided
```

**Numeric fields** (coordinates, charge, mass, residue/local ids):
```python
"x": (lo, hi)                                # bare pair — single include range
"x": ((lo, hi), (exc_lo, exc_hi))            # include range + exclude range
```

Either bound can be `None` for open-ended ranges. **Exclusion always wins over inclusion.**


## 3. Typing Queries  (PDB)

Typing queries filter atoms by name, residue, segment, coordinates, etc.


In [3]:
# --- All atoms (empty query) ---
all_ids = s.get_types(QUERY={}, REQUEST='global_ids')
print('All atoms:          ', all_ids)           # [0..19]

# --- Select by atom name ---
ca_ids = s.get_types(
    QUERY={'atom_name': {'CA'}},
    REQUEST='global_ids',
)
print('CA atoms:           ', ca_ids)            # [2, 8, 12]  (ALA-CA, GLY-CA, PRO-CA)

# --- Select by segment ---
solv_ids = s.get_types(
    QUERY={'segment_name': 'SOLV'},
    REQUEST='global_ids',
)
print('SOLV segment:       ', solv_ids)          # [14..19]

# --- Coordinate range: protein atoms with x <= 5.0 ---
near_ids = s.get_types(
    QUERY={'x': (None, 5.0)},
    REQUEST='global_ids',
)
print('x <= 5.0:           ', near_ids)

# --- Combine filters: backbone N atoms in PROT ---
backbone_n = s.get_types(
    QUERY={
        'atom_name': ({'N', 'CA', 'C', 'O'}, set()),
        'segment_name': 'PROT',
    },
    REQUEST='global_ids',
)
print('Backbone heavy (PROT):', backbone_n)

# --- Exclude a residue: all PROT atoms except GLY ---
no_gly = s.get_types(
    QUERY={
        'segment_name': 'PROT',
        'residue_name': (set(), {'GLY'}),
    },
    REQUEST='global_ids',
)
print('PROT excluding GLY: ', no_gly)            # ALA + PRO atoms


All atoms:           [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
CA atoms:            [2, 8, 12]
SOLV segment:        [14, 15, 16, 17, 18, 19]
x <= 5.0:            [0, 1, 2, 3, 4, 5, 6, 7]
Backbone heavy (PROT): [0, 2, 4, 5, 6, 8, 9, 10, 11, 12, 13]
PROT excluding GLY:  [0, 1, 2, 3, 4, 5, 11, 12, 13]


### Requesting property values

You can also pull per-atom property lists directly from the typing file.


In [4]:
# Residue names of all CA atoms
ca_resnames = s.get_types(
    QUERY={'atom_name': 'CA'},
    REQUEST='residue_names',
)
print('Residue names of CA atoms:', ca_resnames)  # ['ALA', 'GLY', 'PRO']

# x-coordinates of all SOLV atoms
solv_x = s.get_types(
    QUERY={'segment_name': 'SOLV'},
    REQUEST='x',
)
print('SOLV x-coords:', solv_x)


Residue names of CA atoms: ['ALA', 'GLY', 'PRO']
SOLV x-coords: [12.0, 12.9, 11.3, 14.0, 14.8, 13.4]


## 4. Topology Queries  (PSF)

Topology queries can filter by atom type, charge, mass, and by bond graph.
The bond graph is only available via topology — there is no equivalent in the PDB typing file.


In [5]:
# --- All atoms via topology ---
all_topo = s.get_topology(QUERY={}, REQUEST='global_ids')
print('All atoms:              ', all_topo)

# --- Filter by residue name ---
ala_ids = s.get_topology(
    QUERY={'residue_name': {'ALA'}},
    REQUEST='global_ids',
)
print('ALA residue:            ', ala_ids)       # [0..5]

# --- Filter by charge range: negatively charged atoms ---
neg_charged = s.get_topology(
    QUERY={'charge': (None, -0.2)},
    REQUEST='global_ids',
)
print('Charge <= -0.2:         ', neg_charged)

# --- Filter by atom type: all carbon types ---
carbons = s.get_topology(
    QUERY={'atom_type': {'CT1', 'CT2', 'CT3', 'CP1', 'C'}},
    REQUEST='global_ids',
)
print('Carbon atoms:           ', carbons)

# --- Fetch charges for all protein atoms ---
prot_charges = s.get_topology(
    QUERY={'segment_name': 'PROT'},
    REQUEST='charges',
)
print('PROT charges:           ', [round(c, 3) for c in prot_charges])


All atoms:               [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
ALA residue:             [0, 1, 2, 3, 4, 5]
Charge <= -0.2:          [0, 3, 5, 6, 10, 11, 14, 17]
Carbon atoms:            [2, 3, 4, 8, 9, 12, 13]
PROT charges:            [-0.47, 0.31, -0.02, -0.27, 0.51, -0.51, -0.47, 0.31, -0.18, 0.51, -0.51, -0.29, -0.02, 0.51]


### Bond graph queries

The `bonded_with` keyword lets you select atoms based on their chemical environment.
Each constraint block describes a set of required neighbours.


In [6]:
# --- Degree-1 atoms (terminal: HN, CB, O, water Hs, PRO-C) ---
terminal = s.get_topology(
    QUERY={
        'bonded_with': (
            [{'total': True, 'count': {'eq': 1}}],
            [],
        ),
        'bonded_with_mode': ('all', None),
    },
    REQUEST='global_ids',
)
print('Terminal atoms (degree-1):', terminal)     # 10 atoms

# --- Degree-3 atoms (branching: ALA-CA, ALA-C, GLY-N, GLY-C) ---
branching = s.get_topology(
    QUERY={
        'bonded_with': (
            [{'total': True, 'count': {'eq': 3}}],
            [],
        ),
        'bonded_with_mode': ('all', None),
    },
    REQUEST='global_ids',
)
print('Branching atoms (degree-3):', branching)   # [2, 4, 6, 9]

# --- Carbonyl carbons: bonded to exactly one O (type O) and two others ---
# C type is 'C', and carbonyl O has type 'O'
carbonyl_c = s.get_topology(
    QUERY={
        'bonded_with': (
            [
                {'neighbor': {'atom_type': ({'O'}, set())}, 'count': {'eq': 1}},
                {'total': True, 'count': {'eq': 3}},
            ],
            [],
        ),
        'bonded_with_mode': ('all', None),
    },
    REQUEST='global_ids',
)
print('Carbonyl carbons:         ', carbonyl_c)   # ALA-C (4), GLY-C (9), PRO-C (13)

# --- Water oxygens: bonded to exactly 2 HT atoms ---
water_o = s.get_topology(
    QUERY={
        'bonded_with': (
            [{'neighbor': {'atom_type': ({'HT'}, set())}, 'count': {'eq': 2}}],
            [],
        ),
        'bonded_with_mode': ('all', None),
    },
    REQUEST='global_ids',
)
print('Water oxygens:            ', water_o)      # [14, 17]


Terminal atoms (degree-1): [1, 3, 5, 7, 10, 13, 15, 16, 18, 19]
Branching atoms (degree-3): [2, 4, 6, 9]
Carbonyl carbons:          [4, 9]
Water oxygens:             [14, 17]


## 5. Trajectory Queries  (DCD)

Trajectory queries take pre-resolved `global_ids` and a `frame_interval`, and return a numpy array of shape `(n_frames, n_atoms, 3)`.

`frame_interval` is `(start, stop, step)` where both bounds are **inclusive**.


In [7]:
# --- All CA atoms, all 5 frames ---
ca_ids = s.get_types(QUERY={'atom_name': 'CA'}, REQUEST='global_ids')
print('CA global ids:', ca_ids)  # [2, 8, 12]

pos, meta = s.get_trajectory(
    QUERY={
        'global_ids': (ca_ids, set()),
        'frame_interval': (0, 4, 1),
    },
    REQUEST='positions',
)

print('Shape:       ', pos.shape)          # (5, 3, 3)  — 5 frames, 3 CA atoms
print('Dtype:       ', pos.dtype)          # float32
print('Frames read: ', meta['n_frames_read'])
print()
print('CA x-coordinates across frames (x shifts +1 Å per frame):')
for fi in range(pos.shape[0]):
    print(f'  frame {fi}: x = {pos[fi, :, 0].tolist()}')


CA global ids: [2, 8, 12]
Shape:        (5, 3, 3)
Dtype:        float32
Frames read:  5

CA x-coordinates across frames (x shifts +1 Å per frame):
  frame 0: x = [2.4000000953674316, 5.800000190734863, 9.5]
  frame 1: x = [3.4000000953674316, 6.800000190734863, 10.5]
  frame 2: x = [4.400000095367432, 7.800000190734863, 11.5]
  frame 3: x = [5.400000095367432, 8.800000190734863, 12.5]
  frame 4: x = [6.400000095367432, 9.800000190734863, 13.5]


In [8]:
# --- Every other frame (step=2) ---
pos_skip, meta_skip = s.get_trajectory(
    QUERY={
        'global_ids': (ca_ids, set()),
        'frame_interval': (0, 4, 2),
    },
    REQUEST='positions',
)
print('Every 2nd frame shape:', pos_skip.shape)    # (3, 3, 3) — frames 0, 2, 4

# --- Single frame only ---
pos_f2, _ = s.get_trajectory(
    QUERY={
        'global_ids': (ca_ids, set()),
        'frame_interval': (2, 2, 1),
    },
    REQUEST='positions',
)
print('Single frame shape:   ', pos_f2.shape)      # (1, 3, 3)
print('Frame 2 CA positions:\n', pos_f2[0])


Every 2nd frame shape: (3, 3, 3)
Single frame shape:    (1, 3, 3)
Frame 2 CA positions:
 [[ 4.4  5.   5. ]
 [ 7.8  3.2  4.1]
 [11.5  2.8  3.6]]


## 6. The `positions()` Pipeline

`positions()` is the main user-facing method. It combines a typing and/or topology selection with a trajectory read in one call.

```
TYPE_Q  ──┐
          ├─ intersection ─► global_ids ─► DCD read ─► positions array
TOPO_Q  ──┘
```


In [9]:
# --- Select CA atoms via typing, read all 5 frames ---
out = s.positions(
    TYPE_Q={'atom_name': 'CA'},
    TRAJ_Q={'frame_interval': (0, 4, 1)},
)

print('Mode:            ', out['mode'])
print('Atoms selected:  ', out['selection']['count'])   # 3
print('Sources:         ', out['selection']['sources'])  # ['typing']

pos, meta = out['payload']
print('Position shape:  ', pos.shape)           # (5, 3, 3)


Mode:             positions
Atoms selected:   3
Sources:          ['typing']
Position shape:   (5, 3, 3)


In [10]:
# --- Intersect typing and topology selections ---
# TYPE_Q: CA atoms            → [2, 8, 12]
# TOPO_Q: PROT segment        → [0..13]
# Intersection                → [2, 8, 12]  (all CA are in PROT anyway)
#
# More selective intersection:
# TYPE_Q: CA atoms            → [2, 8, 12]
# TOPO_Q: ALA residue only    → [0..5]
# Intersection                → [2]  (only ALA-CA)

out_intersect = s.positions(
    TYPE_Q={'atom_name': 'CA'},
    TOPO_Q={'residue_name': {'ALA'}},
    TRAJ_Q={'frame_interval': (0, 4, 1)},
)

print('Sources:        ', out_intersect['selection']['sources'])  # ['typing', 'topology']
print('Atoms selected: ', out_intersect['selection']['count'])     # 1
pos2, _ = out_intersect['payload']
print('Shape:          ', pos2.shape)                              # (5, 1, 3)


Sources:         ['typing', 'topology']
Atoms selected:  1
Shape:           (5, 1, 3)


In [11]:
# --- planFlag: estimate cost without reading the DCD ---
plan_out = s.positions(
    TYPE_Q={'atom_name': 'CA'},
    TRAJ_Q={'frame_interval': (0, 4, 1)},
    planFlag=True,
)

print('Payload is None (DCD not opened):', plan_out['payload'] is None)
print('Plan:', plan_out['plan'])


Payload is None (DCD not opened): True
Plan: {'planner_mode': 'header', 'n_atoms': 0, 'n_frames': 5, 'estimated_bytes': 0, 'estimated_mib': 0.0}


## 7. Property Requests via `select()`

Use `select()` for requests that return a scalar property of the system rather than per-atom data or positions.


In [12]:
# Number of atoms in the file
n_atoms = s.select(TYPE_R='property-number_of_atoms')
print('Atoms in PDB:      ', n_atoms['payload']['typing'])   # 20

# Number of distinct residues
n_res = s.select(TYPE_R='property-number_of_residues')
print('Residues in PDB:   ', n_res['payload']['typing'])     # 5

# Number of distinct segments
n_seg = s.select(TYPE_R='property-number_of_segments')
print('Segments in PDB:   ', n_seg['payload']['typing'])     # 2

# Bounding box of all atoms
box = s.select(TYPE_R='property-box_size')
print('Box (xmin,xmax,...):',
      tuple(round(v, 2) for v in box['payload']['typing']))

# Total system charge from PSF
total_q = s.select(TOPO_R='property-system_charge')
print('Total system charge:', round(total_q['payload']['topology'], 4))


Atoms in PDB:       20
Residues in PDB:    5
Segments in PDB:    2
Box (xmin,xmax,...): (0.1, 14.8, 1.9, 6.3, 3.2, 5.5)
Total system charge: -0.59


## 8. Discovering Available Keywords and Requests

Before writing a query, use `print_info()` for a formatted overview, or the programmatic getters if you need the sets in code.


In [13]:
# Programmatic access — useful in scripts and parsers
print('Typing keywords:   ', sorted(s.get_type_keys()))
print('Typing requests:   ', sorted(s.get_type_reqs()))
print()
print('Topology keywords: ', sorted(s.get_topo_keys()))
print('Trajectory keywords:', sorted(s.get_traj_keys()))


Typing keywords:    ['atom_name', 'global_ids', 'local_ids', 'occupancy', 'residue_ids', 'residue_name', 'segment_name', 'temperature_coeff', 'x', 'y', 'z']
Typing requests:    ['atom_names', 'global_ids', 'local_ids', 'occupancy', 'positions', 'property-box_size', 'property-number_of_atoms', 'property-number_of_residues', 'property-number_of_segments', 'residue_ids', 'residue_names', 'segment_names', 'temperature_coeff', 'x', 'y', 'z']

Topology keywords:  ['atom_name', 'atom_type', 'bonded_with', 'bonded_with_mode', 'charge', 'global_ids', 'is_virtual', 'local_ids', 'mass', 'residue_ids', 'residue_name', 'segment_name']
Trajectory keywords: ['frame_interval', 'global_ids']


## 9. Loading Files Without a Full Simulation

You can load any combination of files. If no trajectory is loaded, `positions()` reads directly from the typing or topology file and returns a single-frame array.


In [14]:
# Typing-only sim
s_pdb = Sim(typing=pdb_path)
static_pos = s_pdb.positions(TYPE_Q={'atom_name': 'CA'})
pos_arr = static_pos['payload']          # no (pos, meta) tuple — single numpy array
print('Static positions shape:', pos_arr.shape)   # (1, 3, 3)

# Topology-only sim
s_psf = Sim(topology=psf_path)
charges = s_psf.get_topology(
    QUERY={'segment_name': 'SOLV'},
    REQUEST='charges',
)
print('Water charges:', charges)


Static positions shape: (1, 3, 3)
Water charges: [-0.834, 0.417, 0.417, -0.834, 0.417, 0.417]


## 10. Adding a New Format

The architecture is plugin-based. To add support for a new file format (e.g. XTC trajectories):

1. Write `xtc_parse.py` with the four required functions:
   - `_get_trajectory_keys_reqs_xtc(filepath)`
   - `_plan_trajectory_query_xtc(filepath, query_dictionary, request_string, keywords_available, requests_available)`
   - `_get_trajectory_query_xtc(...)`
   - `_update_trajectory_globals_xtc(filepath)`

2. Add `".xtc"` to `supported_formats` in the trajectory domain registry in `main.py`.

That is it — no other changes needed. See the `TRAJECTORY_FORMAT_BLUEPRINT.md` for the full contract specification.
